# Fitting 01 — Split and Preprocessing Parameters

First step of the Fitting stage. Removes exact duplicates, makes the stratified
train/validation/test split (60/20/20), and fits every preprocessing parameter on the
TRAINING split only: sentinel recode targets, age-conditional income medians, the
dependents median, and the winsorisation bounds. The fitted parameters are saved to
`../artifacts/pipeline_params.json` and the three raw splits to parquet, so that
Fitting 02 can apply the identical treatment to all splits without refitting anything.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# Paths, seed and the column rename map shared by the whole project. The artifacts folder
# collects everything the later notebooks need (parameters, data splits, models).
SEED = 42
DATA_PATH = "../data/cs-training.csv"
ARTIFACTS = Path("../artifacts")
ARTIFACTS.mkdir(exist_ok=True)
PAST_DUE_COLS = ["past_due_30_59", "past_due_60_89", "past_due_90"]
RENAME = {
    "SeriousDlqin2yrs": "target",
    "RevolvingUtilizationOfUnsecuredLines": "revolving_utilisation",
    "age": "age",
    "NumberOfTime30-59DaysPastDueNotWorse": "past_due_30_59",
    "DebtRatio": "debt_ratio",
    "MonthlyIncome": "monthly_income",
    "NumberOfOpenCreditLinesAndLoans": "open_credit_lines_and_loans",
    "NumberOfTimes90DaysLate": "past_due_90",
    "NumberRealEstateLoansOrLines": "real_estate_loans_or_lines",
    "NumberOfTime60-89DaysPastDueNotWorse": "past_due_60_89",
    "NumberOfDependents": "dependents",
}

In [3]:
# Load the raw dataset and remove exact duplicate rows (identical on all 11 columns)
# BEFORE splitting. If duplicates were kept, the same record could land in both the
# training and the test split and leak information into the final evaluation.
df = pd.read_csv(DATA_PATH, index_col=0).rename(columns=RENAME)
n_raw = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"raw rows: {n_raw:,}   duplicates removed: {n_raw - len(df):,}   remaining: {len(df):,}")

raw rows: 150,000   duplicates removed: 609   remaining: 149,391


In [4]:
# Stratified 60/20/20 train/validation/test split. Stratifying on the target keeps the
# 6.7% default rate identical across splits, so that model comparison is not distorted by
# prevalence differences. Two chained splits: 60/40 first, then the 40% half into 20/20.
train_df, rest = train_test_split(df, train_size=0.60, stratify=df["target"],
                                  random_state=SEED)
val_df, test_df = train_test_split(rest, train_size=0.50, stratify=rest["target"],
                                   random_state=SEED)
splits = {"train": train_df, "val": val_df, "test": test_df}
pd.DataFrame({name: {"rows": len(part), "default_rate": part["target"].mean()}
              for name, part in splits.items()}).T.round(4)

,rows,default_rate
train,89634.0,0.067
val,29878.0,0.067
test,29879.0,0.067


In [5]:
# Fit the sentinel recode targets on the TRAINING split: each past-due column's 96/98
# codes will be replaced by that column's legitimate maximum + 1, so the sentinel rows
# (default rate ~55%, see EDA 01) stay at the high-risk end of the ordinal count scale
# instead of being deleted or imputed to a harmless value.
sentinel_recode = {
    col: int(train_df.loc[~train_df[col].isin([96, 98]), col].max()) + 1
    for col in PAST_DUE_COLS
}
print("sentinel recode targets (legitimate max + 1):", sentinel_recode)

sentinel recode targets (legitimate max + 1): {'past_due_30_59': 14, 'past_due_60_89': 12, 'past_due_90': 18}


In [6]:
# Fit the imputation parameters on the TRAINING split (after recoding the sentinels, so
# they cannot distort the statistics). Monthly income gets one median per age decile
# because missingness rises with age (EDA 02) and income follows a career arc - a single
# global median would be wrong for both the young and the old. Dependents (2.6% missing)
# gets the global median.
#
# The income medians are computed per integer age-decile CODE (pd.cut labels=False), so the
# stored list is indexed by bin code 0..n_bins-1, the exact code the apply side reconstructs.
# np.unique can collapse tied quantile edges to fewer than 10 bins; keying by code (and
# filling any code unseen in training with the global median) keeps the list length equal to
# the number of bins and immune to the positional-misalignment that a groupby(observed=True)
# list would suffer if a bin were empty.
train_recoded = train_df.copy()
for col, val in sentinel_recode.items():
    train_recoded[col] = train_recoded[col].replace({96: val, 98: val})

age_edges = np.unique(np.quantile(train_recoded["age"], np.linspace(0, 1, 11)))
n_age_bins = len(age_edges) - 1
age_decile_code = pd.cut(train_recoded["age"], bins=age_edges,
                         include_lowest=True, labels=False).astype(int)
global_income_median = float(train_recoded["monthly_income"].median())
income_by_code = train_recoded.groupby(age_decile_code)["monthly_income"].median()
income_medians = [float(income_by_code.get(i, global_income_median)) for i in range(n_age_bins)]
imputation = {
    "age_decile_edges": [float(e) for e in age_edges],
    "income_medians_per_decile": income_medians,
    "dependents_median": float(train_recoded["dependents"].median()),
}
assert len(imputation["income_medians_per_decile"]) == n_age_bins
print("income medians per age decile:", [round(m) for m in imputation["income_medians_per_decile"]])
print("dependents median:", imputation["dependents_median"])

income medians per age decile: [3300, 5000, 5800, 6180, 6400, 6300, 6324, 6000, 5358, 4315]
dependents median: 0.0


In [7]:
# Fit the winsorisation bounds (1st / 99th percentile) on the recoded + imputed TRAINING
# split, for the seven continuous features. Capping extreme values stabilises the
# transformed features used by configuration B; the past-due columns are deliberately NOT
# winsorised because their recoded sentinel values must stay above the legitimate range.
train_imputed = train_recoded.copy()
income_median_by_code = dict(enumerate(imputation["income_medians_per_decile"]))
decile_index = pd.cut(train_imputed["age"].clip(age_edges[0], age_edges[-1]),
                      bins=age_edges, include_lowest=True, labels=False).astype(int)
assert decile_index.between(0, len(income_median_by_code) - 1).all(), \
    "age decile code out of range when imputing income"
income_fill = decile_index.map(income_median_by_code)
train_imputed["monthly_income"] = train_imputed["monthly_income"].fillna(income_fill)
train_imputed["dependents"] = train_imputed["dependents"].fillna(imputation["dependents_median"])

WINSORISE_FEATURES = ["revolving_utilisation", "age", "debt_ratio", "monthly_income",
                      "open_credit_lines_and_loans", "real_estate_loans_or_lines", "dependents"]
winsorisation = {
    col: {"p01": float(train_imputed[col].quantile(0.01)),
          "p99": float(train_imputed[col].quantile(0.99))}
    for col in WINSORISE_FEATURES
}
pd.DataFrame(winsorisation).T.round(2)

,p01,p99
revolving_utilisation,0.0,1.10
age,24.0,87.00
debt_ratio,0.0,4977.67
monthly_income,0.0,23750.00
open_credit_lines_and_loans,0.0,25.00
real_estate_loans_or_lines,0.0,4.00
dependents,0.0,4.00


In [8]:
# Persist everything the next notebooks need: the fitted parameters (one JSON) and the
# three raw splits (parquet). From here on, no parameter is ever fitted on validation or
# test data - Fitting 02 only APPLIES this file.
#
# The transform map and the encoding edges are design constants (the EDA 03 decisions),
# not fitted quantities, but they are recorded here so that pipeline_params.json is a
# complete, self-contained description of the pipeline: a fresh process can rebuild every
# feature configuration from this file plus a raw split, without reading any notebook.
# (The two train-fitted synthetic-feature parameters - Mahalanobis mean/inverse-covariance
# and the per-feature 99th percentiles - are appended to this same file by Fitting 02,
# where the wide training frame they need is built.)
TRANSFORM_MAP = {
    "monthly_income": "sqrt",
    "revolving_utilisation": "log1p",
    "debt_ratio": "log1p",
    "open_credit_lines_and_loans": "log1p",
    "dependents": "log1p",
    "age": "identity",
    "real_estate_loans_or_lines": "identity",
    "past_due_30_59": "identity",
    "past_due_60_89": "identity",
    "past_due_90": "identity",
}
ENCODING = {
    "revolving_util_band_edges": [0.0, 0.30, 0.60, 1.00],
    "age_lifestage_edges": [39, 65],
}
params = {
    "seed": SEED,
    "sentinel_recode": sentinel_recode,
    "imputation": imputation,
    "winsorisation": winsorisation,
    "transform_map": TRANSFORM_MAP,
    "encoding": ENCODING,
}
(ARTIFACTS / "pipeline_params.json").write_text(json.dumps(params, indent=2))
for name, part in splits.items():
    part.reset_index(drop=True).to_parquet(ARTIFACTS / f"{name}.parquet")
print("saved: pipeline_params.json (params + transform_map + encoding), "
      "train.parquet, val.parquet, test.parquet")

saved: pipeline_params.json (params + transform_map + encoding), train.parquet, val.parquet, test.parquet
